In [1]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd

from datetime import datetime
from dateutil.relativedelta import relativedelta
from datetime import datetime, timedelta
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [5]:
from utils.generic_utils import SQLModule
from utils.data_utils import StockPriceProcess
from utils.constants import CURRENCY_MAPPER

In [6]:
CODES = {
    'vietnam' : ['ACB'],
    'australia' : ['TPG', 'TNE','SGLLV']
}
CODES_TO_COUNTRY = {v : k for k,vv in CODES.items() for v in vv}
DAYS = 365
# Fibonacci ratios
RATIOS = [0.236, 0.382, 0.5, 0.618, 0.786]
COLORS = ["black","cyan","magenta","yellow", "purple"]

In [7]:
# Get 1 year data
end_date = datetime.today().date()
start_date = end_date - timedelta(days = DAYS)

In [8]:
fig = make_subplots(
    rows = len(CODES_TO_COUNTRY), 
    cols = 1, 
    subplot_titles = [f'[{country}] {stock_code}' for stock_code, country in CODES_TO_COUNTRY.items()]
)
fig.update_annotations(font = dict(size = 20))
for num,(stock_code,country) in enumerate(CODES_TO_COUNTRY.items()):
    engine = SQLModule.get_engine(country = country)
    query = f"""
        SELECT
            date,
            open,
            high,
            low,
            close
        FROM transaction
        WHERE 
            stock_code = '{stock_code}'
            AND
            date >= DATE '{start_date}'
            AND
            date <= DATE '{end_date}'
        ORDER BY date
    """
    df = pd.read_sql_query(query, engine)
    df = df.set_index('date')
    df = StockPriceProcess.remove_invalid_data(df, country = country)

    # calculate highest and lowest swing
    highest_swing_idx = -1
    lowest_swing_idx = -1
    for i in range(1, len(df) - 1):
        if df.iloc[i]['high'] > df.iloc[i - 1]['high'] and \
            df.iloc[i]['high'] > df.iloc[i + 1]['high'] and \
                (highest_swing_idx == -1 or df.iloc[i]['high'] > df.iloc[highest_swing_idx]['high']):
                highest_swing_idx = i

        if df.iloc[i]['low'] < df.iloc[i - 1]['low'] and \
            df.iloc[i]['low'] < df.iloc[i + 1]['low'] and \
                (lowest_swing_idx == -1 or df.iloc[i]['low'] < df.iloc[lowest_swing_idx]['low']):
                lowest_swing_idx = i
    levels = []

    # calculate levels
    max_level = df.iloc[highest_swing_idx]['high']
    min_level = df.iloc[lowest_swing_idx]['low']

    for ratio in RATIOS:
        if highest_swing_idx > lowest_swing_idx: # Uptrend
            levels.append(max_level - (max_level - min_level) * ratio)
        else: #Downtrend
            levels.append(min_level + (max_level - min_level) * ratio)

    fig.add_trace(go.Scatter(
        x = df.index, 
        y = df['close'], 
        name = stock_code,
        hoverinfo='x+y',
        marker = dict(color = 'blue'),
        showlegend = False
    ), row = num + 1, col = 1)

    fig.update_yaxes(
        tickprefix = f'{CURRENCY_MAPPER[country]} ',
        row = num + 1, col = 1
    )

    for i in range(len(levels)):
        fig.add_trace(go.Scatter(
            x = [df.index[0], df.index[-1]], 
            y = [levels[i], levels[i]], 
            name = stock_code,
            hoverinfo='y',
            marker = dict(color = COLORS[i]),
            showlegend = False
        ), row = num + 1, col = 1)

fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    height = 400 * len(CODES_TO_COUNTRY),
    font = dict(size = 20),
)

fig.show()